# 1. RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# 2. [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 2.1 설치
- `pip install ragas rapidfuzz`

## 2.2 RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 2.3 주요 평가지표
### 2.3.1 Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### 2.3.1.1 Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 2.3.1.1.1평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### 2.3.2 Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 2.3.2.1 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## 2.4 Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### 2.4.1 Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 2.4.1.1평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 2.4.1.2 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### 2.4.2 Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 2.4.2.1 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# 3. RAGAS 평가 실습

In [ ]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

Resolved 93 packages in 1.89s
 Downloaded rapidfuzz
 Downloaded scikit-network
Prepared 8 packages in 530ms
Uninstalled 3 packages in 89ms
Installed 11 packages in 453ms
 + appdirs==1.4.4
 + datasets==5.0.0
 + diskcache==5.6.3
 - fsspec==2026.6.0
 + fsspec==2026.4.0
 + instructor==1.15.4
 - jiter==0.15.0
 + jiter==0.14.0
 + nest-asyncio==1.6.0
 + ragas==0.4.3
 + rapidfuzz==3.14.5
 - rich==15.0.0
 + rich==14.3.4
 + scikit-network==0.33.5


In [ ]:
# docker run -p 6333:6333 -p 6334:6334   -v qdrant_storage:/qdrant/storage   qdrant/qdrant

In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, SparseVectorParams, VectorParams
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv

load_dotenv()


True

In [2]:
# ##############################################################
# 데이터 준비
##############################################################

def load_and_split_olympic_data(file_path="data/olympic_wiki.md"):
    with open(file_path, "r", encoding="utf-8") as fr:
        olympic_text = fr.read()

    # Split
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "H1"),
            ("##", "H2"),
            ("###", "H3"),
        ],
    )

    return splitter.split_text(olympic_text)

In [4]:
#################################################################
# Vector DB 연결
# retriever 생성
#################################################################

def get_vectorstore(collection_name: str = "olympic_info_wiki"):


    dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

    client = QdrantClient(url="http://localhost:6333")

    # 컬렉션 삭제
    if client.collection_exists(collection_name):
        result = client.delete_collection(collection_name=collection_name)

    # 컬렉션 생성
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
      
    )

    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,    
        embedding=dense_embeddings
    )
    
    ######################################
    # Document들 추가
    ######################################
    documents = load_and_split_olympic_data()
    vectorstore.add_documents(documents=documents)

    return vectorstore


def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

In [5]:
vectorstore = get_vectorstore()

retriever = get_retriever(vectorstore)
retriever

VectorStoreRetriever(tags=['QdrantVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x0000027BB7DCBB60>, search_kwargs={'k': 5})

In [7]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from operator import itemgetter

vectorstore = get_vectorstore()
retriever = get_retriever(vectorstore)

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(
    template=prompt_txt
)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

def format_doc_to_str(documents:list[Document])->list[str]:
    """
    VectorStore에 조회한 문서들(list[Document])에서 내용(page_content)만 추출해서 list[str] 로 반환.
    RAGAS 평가시 context는 각 검색한 문서를 list[str] 로 받기 때문에 이렇게 처리.
    
    Args:
        documents(list[Document]): [Document(..), Document(...), ..]}
    Returns:
        list[str]: 각 문서의 내용만 추출해서 리스트에 담는다.
    """
    return [doc.page_content for doc in documents]

# RAG 체인 -> RAG 평가 데이터셋을 만드는 RAG Chain -> 최종응답: LLM의 응답(str), 검색한 문서들(list[str])
chain = RunnablePassthrough() | {
    "context":retriever | format_doc_to_str,
    "query":RunnablePassthrough()
} | { 
    "response": prompt | model | parser, # LLM 응답
    "retrieved_context": itemgetter("context") # 검색한 문서들 반환
}
# RunnablePassthrough() -> LCEL 체인을 만들려면 구성요소중 하나가 Runnable이어야 하는데 이 체인은 dicr |dict 구조이기 때문에 추가. 
# 추가하지 않으면  | 연산(or 연산이 된다)

In [8]:
res = chain.invoke("1회 올림픽은 언제 어디서 열렸지")

In [13]:
print(res.keys())
print(res)

dict_keys(['response', 'retrieved_context'])
{'response': '정보가 부족해 답을 할 수없습니다.', 'retrieved_context': ['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에

In [16]:
res["response"]

'정보가 부족해 답을 할 수없습니다.'

In [15]:
res["retrieved_context"]

['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스 

In [17]:
res

{'response': '정보가 부족해 답을 할 수없습니다.',
 'retrieved_context': ['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요

In [ ]:
# !uv pip install langchain_google_vertexai

Resolved 63 packages in 989ms
 Downloaded pyarrow
 Downloaded google-cloud-aiplatform
Prepared 20 packages in 4.12s
Uninstalled 2 packages in 271ms
Installed 20 packages in 1.30s
 + bottleneck==1.6.0
 + google-api-core==2.31.0
 + google-cloud-aiplatform==1.159.0
 + google-cloud-bigquery==3.42.1
 + google-cloud-core==2.6.0
 + google-cloud-resource-manager==1.18.0
 + google-cloud-storage==3.12.0
 + google-cloud-vectorsearch==0.11.1
 + google-crc32c==1.8.0
 + google-resumable-media==2.10.0
 + googleapis-common-protos==1.75.0
 + grpc-google-iam-v1==0.14.4
 + grpcio-status==1.81.1
 + langchain-google-vertexai==3.2.4
 + numexpr==2.14.1
 + proto-plus==1.28.0
 - protobuf==7.35.1
 + protobuf==6.33.6
 - pyarrow==24.0.0
 + pyarrow==23.0.1
 + pyopenssl==26.3.0
 + validators==0.35.0


# 4. RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## 4.1 TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.


> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>           from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [ ]:
# 주피터노트북 환경에서 비동기적 처리 위해 
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [20]:
from ragas.testset import TestsetGenerator

In [ ]:
#
# testset -> Context들(문서들) - [질문 - 정답답변 + (Retriever가 찾은 문서 + LLM 응답: chain에서 생성)]
# 1. Context(문서들)을 추출 - TestsetGenerator -> 질문과 정답 답변 생성

import random

# 데이터셋을 생성할 때 사용할 Context를 추출
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "olympic_info_wiki"

info = client.get_collection(COLLECTION_NAME)
total_docs = info.points_count

results, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=total_docs,
)

# random하게 k(5)개를 sampling
sample_docs:"list[PointStruct]" =random.sample(results, 5) # 리스트에서 랜덤하게 k(5)개를 추출

# PointStruct - payload: page_content, metadata
# page_content만 추출해서 list[str]
docs = [point.payload['page_content'] for point in sample_docs]



In [24]:
total_docs

25

In [23]:
docs

["올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽기 게양에 이어 올림픽 선서를 끝으로 개막식 과정은 모두 끝나게 된다. 2020년 하계 올림픽 이후로는 그리스 - 난민 올림픽 선수단 - 나머지 각국 선수단 - 차차기 올림픽 개최국 - 차기 올림픽 개최국 - 개최국 순서대로 입장한다.  \n#### 폐막식  \n폐막식은 올림픽 경기가 모두 끝난 후에 열린다. 각국의 기수가 경기장에 들어온 후에 국적에 상관

In [26]:
# testset 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#testsetgenerator 는 gpt-5 이후 버전은 사용할 수 없다. 
## Langchain의 모델과 Embedding 모델 -> ragas에서 사용할 수 있도록 변환(wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 사람들이 올림픽에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법읋 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" # 질문/답변을 생성할 때 LLM에게 전달할 system Prompt를 설정

)





C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\833878933.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\833878933.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [ ]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용 테스트셋 개수(질문 답변 개수)
)

Applying SummaryExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Node 1ce75824-fa9e-46c6-a23f-71ee3698094f does not have a summary. Skipping filtering.
Node a0b21374-443a-4200-8337-f6fa0b4a7b82 does not have a summary. Skipping filtering.
Node fa7cb2a9-bd91-4334-8c71-4651c019b025 does not have a summary. Skipping filtering.
Node 43fc9ce4-607b-43d0-8205-20d26782ef02 does not have a summary. Skipping filtering.
Node c78441a0-03aa-46c8-b12e-981e0159ab6f does not have a summary. Skipping filtering.


Applying EmbeddingExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [30]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='그리스 언제 입장해요?', retrieved_contexts=None, reference_contexts=["올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽기 게양에 이어 올림픽 선서를 끝으로 개막식 과정은 모두 끝나게 된다. 2020년 하계 올림픽 이후로는 그리스 - 난민 올림

In [39]:
sample1 = testset.samples[0].eval_sample
print("사용자질문:", sample1.user_input)
print("Context:", sample1.reference_contexts)

print("생성된 답변(정답):", sample1.reference)
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)


사용자질문: 그리스 언제 입장해요?
Context: ["올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽기 게양에 이어 올림픽 선서를 끝으로 개막식 과정은 모두 끝나게 된다. 2020년 하계 올림픽 이후로는 그리스 - 난민 올림픽 선수단 - 나머지 각국 선수단 - 차차기 올림픽 개최국 - 차기 올림픽 개최국 - 개최국 순서대로 입장한다.  \n#### 폐막식  \n폐막식은 올림픽 경기가 모두 끝난 후에 열린

In [40]:
# 생성된 Testset을 Pandas DataFrame으로 변환
eval_df = testset.to_pandas()

In [42]:
eval_df.shape

(10, 7)

In [43]:
eval_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,그리스 언제 입장해요?,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",올림픽 개막식에서 그리스는 전통적으로 맨 처음에 입장합니다. 2020년 하계 올림픽...,Paralympic Sports Enthusiast,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,이탈리아 코르티나담페초에서 열릴 예정이었던 올림픽은 어떤 이유로 취소되었나요?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",이탈리아 코르티나담페초에서 열릴 예정이었던 1944년 동계 올림픽은 제2차 세계대전...,Olympic Historian,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,패럴림픽의 시작과 발전에 있어 제2차 세계대전이 어떤 역할을 했는지 설명해 주세요.,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,패럴림픽은 제2차 세계대전에 참전한 군인들의 사회 복귀를 돕기 위한 목적으로 시작되...,Paralympic Sports Enthusiast,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
3,국제올림픽위원회가 하는일이 뭐에요?,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시...",Olympic Historian,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
4,올림픽이 상업성과 관련하여 어떤 비판을 받고 있습니까?,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...,올림픽은 지나치게 상업성과 연계되어 기존의 상업적인 스포츠쇼와 다를 게 없다는 비판...,Olympic Event Coordinator,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer


In [44]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']
resp = chain.invoke(q)

In [45]:
resp['response']

'그리스는 올림픽 개막식에서 전통적으로 **맨 처음에 입장**합니다.'

In [46]:
resp['retrieved_context']

['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스 

In [47]:
# 모든 testset의 데이터데 대한 llm 응답과 retriever의 검색 겷과를 추가
response_list = []
retrieved_context_list = []

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp['response'])
    retrieved_context_list.append(resp['retrieved_context'])

In [50]:
print(len(response_list), len(retrieved_context_list))

10 10


In [51]:
response_list

['그리스는 올림픽 개막식 선수단 입장 때 전통적으로 맨 처음에 입장합니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '제공된 Context에 따르면, 제2차 세계대전은 패럴림픽의 시작에 직접적인 계기가 되었습니다.\n\n- 1948년, 루드비히 구트만 경은 **제2차 세계대전에 참전한 군인들의 사회 복귀**를 돕기 위한 일환으로, 런던 올림픽과 동시에 여러 병원들이 연합해 경기를 열었습니다.\n- 이 대회는 **세계 휠체어, 신체부자유자대회(World Wheelchair and Amputee Games)**로 알려졌고, 이후 매년 열리는 스포츠대회로 발전했습니다.\n- 이후 구트만과 다른 사람들은 **스포츠를 상처 치유의 방법**으로 보며 대회 개최를 계속 추진했습니다.\n- 그 결과 1960년 로마 하계 올림픽 때 400명의 선수가 참가한 **“Parallel Olympics”**가 열렸고, 이것이 **1회 패럴림픽**으로 알려지게 되었습니다.\n\n즉, 제2차 세계대전은 전쟁에 참전했던 장애 군인들의 재활과 사회 복귀를 위한 스포츠 대회가 만들어지는 계기가 되었고, 이것이 패럴림픽으로 발전했습니다.',
 '국제올림픽위원회(IOC)는 올림픽 활동을 통솔하는 단체입니다.  \n주어진 context에 따르면 IOC의 하는 일은 다음과 같습니다.\n\n- 올림픽 개최 도시를 선정함\n- 올림픽 계획을 감독함\n- 올림픽 종목을 결정하거나 변경함\n- 스폰서 및 방송권 계약을 체결함\n- 올림픽 헌장에 따라 올림픽 전체 활동을 관리함\n\n정보가 부족해 답을 할 수없습니다.',
 '올림픽은 지나치게 상업성과 연계되어 이제는 기존의 상업적인 스포츠쇼와 다를 게 없다는 비판을 받고 있습니다. 또한 개최도시가 올림픽 관련 물건 판매 상인과 회사들로 넘쳐나 시장이 포화되는 문제도 비판받고 있습니다.',
 '1920년 하계 올림픽에서 개막식의 기본 토대가 만들어졌습니다.  \n이때 형성된 주요 전통은 다음과 같습니다.\n\n- 개최국의 국기 게양과 국가 연주로 시작

In [52]:
retrieved_context_list

[['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스

In [69]:
#
# eval_df에 컬럼으로 추가
#
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_context,retrieved_contexts
0,그리스 언제 입장해요?,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",올림픽 개막식에서 그리스는 전통적으로 맨 처음에 입장합니다. 2020년 하계 올림픽...,Paralympic Sports Enthusiast,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,그리스는 올림픽 개막식 선수단 입장 때 전통적으로 맨 처음에 입장합니다.,[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...,[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...
1,이탈리아 코르티나담페초에서 열릴 예정이었던 올림픽은 어떤 이유로 취소되었나요?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",이탈리아 코르티나담페초에서 열릴 예정이었던 1944년 동계 올림픽은 제2차 세계대전...,Olympic Historian,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...","[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실..."
2,패럴림픽의 시작과 발전에 있어 제2차 세계대전이 어떤 역할을 했는지 설명해 주세요.,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,패럴림픽은 제2차 세계대전에 참전한 군인들의 사회 복귀를 돕기 위한 목적으로 시작되...,Paralympic Sports Enthusiast,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer,"제공된 Context에 따르면, 제2차 세계대전은 패럴림픽의 시작에 직접적인 계기가...",[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...
3,국제올림픽위원회가 하는일이 뭐에요?,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시...",Olympic Historian,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer,국제올림픽위원회(IOC)는 올림픽 활동을 통솔하는 단체입니다. \n주어진 cont...,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기..."
4,올림픽이 상업성과 관련하여 어떤 비판을 받고 있습니까?,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...,올림픽은 지나치게 상업성과 연계되어 기존의 상업적인 스포츠쇼와 다를 게 없다는 비판...,Olympic Event Coordinator,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,올림픽은 지나치게 상업성과 연계되어 이제는 기존의 상업적인 스포츠쇼와 다를 게 없다...,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...
5,"1920년 하계 올림픽에서 만들어진 개막식과 폐막식의 주요 전통들은 무엇이며, 이 ...","[<1-hop>\n\n올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 ...","1920년 하계 올림픽(벨기에 안트베르펜)에서 개막식의 기본 토대가 만들어졌으며, ...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer,1920년 하계 올림픽에서 개막식의 기본 토대가 만들어졌습니다. \n이때 형성된 ...,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...","[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#..."
6,"1920년 하계 올림픽이 올림픽 개막식과 폐막식의 전통에 어떤 영향을 미쳤으며, 이...","[<1-hop>\n\n올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 ...",1920년 하계 올림픽은 올림픽 개막식과 폐막식의 전통에 중요한 영향을 미쳤다. 개...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,1920년 하계 올림픽은 **개막식의 기본 토대**를 만든 대회로 설명되어 있습니다...,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...","[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#..."
7,1996년 하계 올림픽 기간에 올림픽 브랜드 상품 시장의 포화와 IOC의 상업성에 ...,[<1-hop>\n\n올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는...,1996년 하계 올림픽 기간에는 올림픽 관련 상품 시장이 포화 상태에 이르러 개최도...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,1996년 하계 올림픽과 2000년 하계 올림픽 사이에 올림픽 관련 상품 시장이 포...,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...
8,"IOC 뭐하는 단체인지랑, 올림픽 상업성 논란에 대해 IOC가 어떻게 대처했는지 알려줘.","[<1-hop>\n\n올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • ...","IOC는 올림픽 활동을 통솔하는 단체로, 올림픽 개최 도시 선정, 계획 감독, 종목...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer,"IOC(국제올림픽위원회)는 **모든 올림픽 활동을 통솔하는 단체**로, \n- *...",[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...
9,1940년 동계 올림픽이 취소된 이유와 이와 관련된 다른 올림픽 취소 사례에는 무엇...,"[<1-hop>\n\n쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져...","1940년 동계 올림픽은 제2차 세계대전으로 인해 취소되었다. 이와 관련하여, 같은...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...","[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실..."


In [68]:
# eval_df를 ragas의 평가데이터셋 타입으로 변환
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=10)

In [ ]:
#
# 평가
#
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    Faithfulness,
    AnswerRelevancy,
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\394508546.py:3: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\394508546.py:3: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\394508546.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\P

In [67]:
from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, Faithfulness,AnswerRelevancy
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\2506188928.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, Faithfulness,AnswerRelevancy
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\2506188928.py:1: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, Faithfulness,AnswerRelevancy
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\2506188928.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be remo

In [72]:
# 평가할 때 사용할 LLM embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)

]
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\384477035.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15044\384477035.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Playdata\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000027B1F74BB00> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Playdata\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000027B1F74BB00> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-4502' coro=<_async_in_context.<locals>.run_in_context() d

In [74]:
eval_result

{'context_recall': 1.0000, 'llm_context_precision_with_reference': 0.8986, 'faithfulness': 0.5000, 'answer_relevancy': 0.4852}

In [75]:
result_df = eval_result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,그리스 언제 입장해요?,[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...,그리스는 올림픽 개막식 선수단 입장 때 전통적으로 맨 처음에 입장합니다.,올림픽 개막식에서 그리스는 전통적으로 맨 처음에 입장합니다. 2020년 하계 올림픽...,1.0,0.2000,NaN,0.277092
1,이탈리아 코르티나담페초에서 열릴 예정이었던 올림픽은 어떤 이유로 취소되었나요?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",정보가 부족해 답을 할 수없습니다.,이탈리아 코르티나담페초에서 열릴 예정이었던 1944년 동계 올림픽은 제2차 세계대전...,1.0,1.0000,NaN,0.000000
2,패럴림픽의 시작과 발전에 있어 제2차 세계대전이 어떤 역할을 했는지 설명해 주세요.,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,"제공된 Context에 따르면, 제2차 세계대전은 패럴림픽의 시작에 직접적인 계기가...",패럴림픽은 제2차 세계대전에 참전한 군인들의 사회 복귀를 돕기 위한 목적으로 시작되...,1.0,1.0000,NaN,0.535222
3,국제올림픽위원회가 하는일이 뭐에요?,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...",국제올림픽위원회(IOC)는 올림픽 활동을 통솔하는 단체입니다. \n주어진 cont...,"국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시...",1.0,0.8875,NaN,0.000000
4,올림픽이 상업성과 관련하여 어떤 비판을 받고 있습니까?,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...,올림픽은 지나치게 상업성과 연계되어 이제는 기존의 상업적인 스포츠쇼와 다를 게 없다...,올림픽은 지나치게 상업성과 연계되어 기존의 상업적인 스포츠쇼와 다를 게 없다는 비판...,1.0,1.0000,NaN,0.995978
5,"1920년 하계 올림픽에서 만들어진 개막식과 폐막식의 주요 전통들은 무엇이며, 이 ...","[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",1920년 하계 올림픽에서 개막식의 기본 토대가 만들어졌습니다. \n이때 형성된 ...,"1920년 하계 올림픽(벨기에 안트베르펜)에서 개막식의 기본 토대가 만들어졌으며, ...",NaN,NaN,NaN,0.877827
6,"1920년 하계 올림픽이 올림픽 개막식과 폐막식의 전통에 어떤 영향을 미쳤으며, 이...","[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",1920년 하계 올림픽은 **개막식의 기본 토대**를 만든 대회로 설명되어 있습니다...,1920년 하계 올림픽은 올림픽 개막식과 폐막식의 전통에 중요한 영향을 미쳤다. 개...,NaN,1.0000,NaN,0.694794
7,1996년 하계 올림픽 기간에 올림픽 브랜드 상품 시장의 포화와 IOC의 상업성에 ...,[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...,1996년 하계 올림픽과 2000년 하계 올림픽 사이에 올림픽 관련 상품 시장이 포...,1996년 하계 올림픽 기간에는 올림픽 관련 상품 시장이 포화 상태에 이르러 개최도...,1.0,1.0000,0.5,0.797298
8,"IOC 뭐하는 단체인지랑, 올림픽 상업성 논란에 대해 IOC가 어떻게 대처했는지 알려줘.",[올림픽 브랜드의 판매는 항상 논란이 되어왔다. 비판 중 하나는 올림픽이 지나치게 ...,"IOC(국제올림픽위원회)는 **모든 올림픽 활동을 통솔하는 단체**로, \n- *...","IOC는 올림픽 활동을 통솔하는 단체로, 올림픽 개최 도시 선정, 계획 감독, 종목...",1.0,1.0000,1.0,0.673842
9,1940년 동계 올림픽이 취소된 이유와 이와 관련된 다른 올림픽 취소 사례에는 무엇...,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",정보가 부족해 답을 할 수없습니다.,"1940년 동계 올림픽은 제2차 세계대전으로 인해 취소되었다. 이와 관련하여, 같은...",1.0,1.0000,0.0,0.000000
